## 02_敘述統計與資料重新分組
本筆記本的目標有兩個：

1. 將自變數 PhysicalFighting 重新分組：原始資料的打架次數過於零散（1~7等代號），為了符合 ANOVA 的分析便利性，我們將其重新編碼為三個離散群體（無衝突、偶爾衝突、頻繁衝突）。

2. 計算分組敘述統計量：算出各組的樣本數、平均數、標準差，初步觀察不同打架頻率學生的螢幕使用時間差異。

📐 打架次數重新分組對照表根據 CDC 2007 說明書，PhysicalFighting 的原始數值與我們預計轉換的群組如下：

* 原始數值 `1`：代表 0 次 $\rightarrow$ 重新編碼為：`1_None`（無衝突組）
* 原始數值 `2, 3, 4`：代表 1~5 次 $\rightarrow$ 重新編碼為：`2_Occasional`（輕度/偶爾衝突組）
* 原始數值 `5, 6, 7`：代表 6 次以上 $\rightarrow$ 重新編碼為：`3_Frequent`（頻繁/高度衝突組）

### 讀取與分組轉換

In [1]:
# ==========================================
# 步驟 1：匯入套件與讀取已清洗資料
# ==========================================
import pandas as pd
import numpy as np

# 讀取我們在 01 檔案產出的乾淨 CSV
input_path = "../data/processed/YRBS_2007_cleaned.csv"
df = pd.read_csv(input_path)
print(f"成功讀取乾淨資料！目前共有 {df.shape[0]} 筆有效學生紀錄。")

# ==========================================
# 步驟 2：定義重新分組的邏輯
# ==========================================
# 我們寫一個小函數，根據原始代號回傳新的組別名稱
def recode_fighting(value):
    if value == 1:
        return "1_None"          # 0次
    elif value in [2, 3, 4]:
        return "2_Occasional"    # 1-5次
    elif value in [5, 6, 7]:
        return "3_Frequent"      # 6次以上
    else:
        return "Unknown"         # 防錯用

# 將這個分組邏輯套用到資料表中，建立一個新欄位叫 'Fighting_Group'
df['Fighting_Group'] = df['PhysicalFighting'].apply(recode_fighting)

print("\n-> 成功建立分組變數 [Fighting_Group]！")
print("前 5 列資料分組預覽：")
display(df[['PhysicalFighting', 'Fighting_Group', 'Total_Screen_Time']].head())

成功讀取乾淨資料！目前共有 13479 筆有效學生紀錄。

-> 成功建立分組變數 [Fighting_Group]！
前 5 列資料分組預覽：


,PhysicalFighting,Fighting_Group,Total_Screen_Time
0,8.0,Unknown,2.5
1,4.0,2_Occasional,3.0
2,1.0,1_None,0.0
3,1.0,1_None,0.0
4,1.0,1_None,4.5


### 計算分組平均數與標準差

In [2]:
# ==========================================
# 步驟 3：計算三組的敘述統計摘要（人數、平均數、標準差）
# ==========================================
print("--- 各打架頻率群體的每日總螢幕時間統計摘要 ---")

# 使用 groupby 針對新組別進行聚合計算
summary_table = df.groupby('Fighting_Group')['Total_Screen_Time'].agg(
    Sample_Size='count',      # 各組人數 (n)
    Mean_Hours='mean',        # 平均螢幕時數 (μ)
    Std_Dev='std'            # 標準差 (sd)
).reset_index()

# 讓表格看起來更精緻：把小數點固定在兩位數
summary_table['Mean_Hours'] = summary_table['Mean_Hours'].round(2)
summary_table['Std_Dev'] = summary_table['Std_Dev'].round(2)

# 顯示這張摘要表
display(summary_table)

--- 各打架頻率群體的每日總螢幕時間統計摘要 ---


,Fighting_Group,Sample_Size,Mean_Hours,Std_Dev
0,1_None,8628,3.65,2.43
1,2_Occasional,4133,3.89,2.51
2,3_Frequent,394,4.45,2.70
3,Unknown,324,4.88,3.25


In [4]:
# =====================================================================
# 步驟 4：剔除 Unknown 無效分組，並輸出最終 ANOVA 專用資料表
# =====================================================================

# 1. 只保留 None, Occasional, Frequent 這三組的學生
df_final_groups = df[df['Fighting_Group'] != "Unknown"].copy()

print(f"原本資料筆數: {df.shape[0]} 筆")
print(f"剔除 Unknown 324 筆後，剩餘有效筆數: {df_final_groups.shape[0]} 筆\n")

# 2. 重新印出完美的統計摘要表確認
print("--- 【修正後】各打架頻率群體的每日總螢幕時間統計摘要 ---")
updated_summary = df_final_groups.groupby('Fighting_Group')['Total_Screen_Time'].agg(
    Sample_Size='count',
    Mean_Hours='mean',
    Std_Dev='std'
).reset_index()

updated_summary['Mean_Hours'] = updated_summary['Mean_Hours'].round(2)
updated_summary['Std_Dev'] = updated_summary['Std_Dev'].round(2)
display(updated_summary)

# 3. 將這個「已分組」的最終資料表，存成另一個新檔案
output_path_g = "../data/processed/YRBS_2007_grouped.csv"
df_final_groups.to_csv(output_path_g, index=False)
print(f"\n 最終分組資料已輸出至：{output_path_g}")

原本資料筆數: 13479 筆
剔除 Unknown 324 筆後，剩餘有效筆數: 13155 筆

--- 【修正後】各打架頻率群體的每日總螢幕時間統計摘要 ---


,Fighting_Group,Sample_Size,Mean_Hours,Std_Dev
0,1_None,8628,3.65,2.43
1,2_Occasional,4133,3.89,2.51
2,3_Frequent,394,4.45,2.70



 最終分組資料已輸出至：../data/processed/YRBS_2007_grouped.csv


### 5. Output (本筆記本輸出說明)

本筆記本最終輸出的檔案為：
`../data/processed/YRBS_2007_grouped.csv`

這個檔案裡多了一個關鍵的欄位：`Fighting_Group`（包含 `1_None`, `2_Occasional`, `3_Frequent` 三個純淨的文字群組），這是專門為了單因子變異數分析 (One-way ANOVA) 所準備的。
